# Plausible Actionable Counterfactual Recourse with Action Paths (PACR-AP)

**Cours** : IADATA708 — Fairness, Interprétabilité, Robustesse<br>
**Dataset** : Statlog (German Credit), Hofmann (1994)<br>
**Modèle** : régression logistique (baseline cost-sensitive)<br>

### Convention d'encodage de l'attribut sensible $A$

| Code $A$ | Groupe | Statut socio-économique | Effectif (n=1000) |
|:---:|:---|:---|:---:|
| **0** | **jeunes** (`age < 25`) | défavorisé | 190 |
| **1** | **adultes** (`age ≥ 25`) | favorisé | 810 |

Les gaps de fairness sont calculés dans le sens **$\Delta_M = M_{\text{jeunes}} - M_{\text{adultes}}$** (défavorisé − favorisé).


## 1. La méthode en bref

**Question** : pour un emprunteur refusé par notre modèle de scoring crédit,
existe-t-il une **séquence d'actions plausibles** qui basculerait la décision ?
Et cette possibilité est-elle équitable entre groupes d'âge ?

**Notre méthode**, *PACR-AP*, part du contrefactuel de Wachter (2017) —
*« trouver un point qui change la décision »* — mais l'adapte à trois
contraintes propres au crédit qui nous obligent à emprunter ailleurs.

**1. Toutes les actions ne sont pas éligibles.** On ne peut pas demander à
un emprunteur de changer son âge ou son sexe. Il faut donc déclarer
explicitement ce qui est modifiable et exiger que le profil final reste
plausible : c'est le **schéma d'actionnabilité** d'**Ustun et al. (2019)**.

**2. Atteindre un profil final plausible ne suffit pas.** Un client ne
change pas toutes ses features d'un coup — il passe par une **séquence
d'actions élémentaires**, et chaque étape intermédiaire doit elle aussi rester
crédible. On emprunte à **FACE (Poyiadzi et al., 2020)** l'idée de raisonner
en **chemins**, avec plausibilité **à chaque étape** et pas seulement à l'arrivée.

**3. Un seul chemin est trop rigide.** Chaque client a ses propres contraintes
— certains peuvent réduire leur montant mais pas leur durée. D'où l'idée de
**DiCE (Mothilal et al., 2020)** : retourner un **top-K diversifié** pour
laisser le choix.

> **À noter** : la méthode répond à *« comment faire basculer la décision
> du modèle ? »*, pas à *« comment réduire le vrai risque de défaut ? »*.
> Les chemins sont **prédictifs**, pas **causaux** — pour ça il faudrait
> un modèle causal structurel (Karimi et al. 2021), hors scope ici.

### Récap : les briques empruntées

| # | Brique | Source |
|---|---|---|
| 1 | Contrefactuel basculant la décision | Wachter, Mittelstadt & Russell (2017) |
| 2 | Schéma d'actionnabilité (mutable / direction) | Ustun, Spangher & Liu (2019) |
| 3 | K chemins divers (top-K avec diversité) | Mothilal, Sharma & Tan (2020) — *DiCE* |
| 4 | Exploration discrète + plausibilité de trajectoire | Karimi et al. (2020) — *MACE* ; Poyiadzi et al. (2020) — *FACE* |
| 5 | Plausibilité conjointe (filtre jointement rares) | Breunig, Kriegel, Ng & Sander (2000) — *LOF* |

### Ce qu'on ajoute en propre

- Seuil de décision **cost-optimal** ($\tau^* = 5/6 \approx 0.833$, matrice de coût German Credit)
- **Magnitudes d'actions minées** depuis les transitions refusé → favorable du training
- **Plausibilité hybride** : marginale (distance NN) **et** jointe (LOF)
- Audit **fairness *of recourse*** par groupe : équité du coût et de la robustesse (distincte des métriques classiques DP / EOpp / PP)

> **Détail formel** : voir [Annexe A](#annexe-A-spécification-formelle-détaillée).
> **Discussion littérature complète** : voir [Annexe B](#annexe-B-littérature-et-emprunts-détaillés).

## 2. Overview du pipeline

Deux niveaux de lecture : ce qu'on fait **pour un emprunteur** refusé, et ce
qu'on fait **au niveau de l'audit** pour juger si la méthode est équitable.

### Pipeline par individu

![Pipeline PACR-AP](figures/pacr_ap_pipeline.png)

Pour chaque emprunteur refusé, on construit un graphe de chemins d'actions
élémentaires (*« réduire le montant de 20 % »*, *« améliorer son compte
courant d'un cran »*) en partant de son profil et en respectant le schéma
d'actionnabilité. À chaque étape, on jette les profils intermédiaires qui ne
ressembleraient pas à un emprunteur réel. Parmi les chemins survivants qui
basculent la décision, on garde **K chemins distincts** qui font le meilleur
compromis entre coût, plausibilité et robustesse — chacun présenté en langage
métier.

### Audit fairness *of recourse*

![Audit fairness PACR-AP](figures/pacr_ap_audit.png)

On rejoue ce pipeline sur tous les refusés et on agrège les résultats par
groupe sensible (jeunes vs adultes) pour calculer les **gaps** de coverage,
coût et robustesse. Comme le schéma d'actionnabilité est un choix politique
discutable, on stress-teste le verdict sous trois variants raisonnables —
il n'est défendable que si les gaps gardent le même signe sur les trois.


## 3. Setup expérimental

Chargement des données (OpenML), entraînement de la **baseline LR cost-sensitive**,
puis instanciation du pipeline PACR-AP. Toute la mécanique est dans le module
[`fairlib.pacr_ap`](../src/fairlib/pacr_ap.py) ; le notebook orchestre.

**Calibrations empiriques** (toutes apprises du training) :
- Magnitudes d'actions par mining
- Seuils de plausibilité $\varepsilon$ (p95 NN) et $\rho_{\text{LOF}}$ (p5 LOF)
- Coûts d'actions : $|\Delta|/\sigma$ pour les continues, $-\log\widehat{P}$ pour les ordinales

**Audit équilibré** : 15 jeunes + 15 adultes parmi les refusés (au lieu des
30 premiers — qui auraient été ~3 jeunes vs ~27 adultes), pour avoir des IC
bootstrap comparables sur les gaps.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import Callable, Optional, Dict, List, Tuple, Any
from collections import Counter, deque
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import roc_auc_score, accuracy_score

SEED = 42
AGE_THRESHOLD = 25
# Matrice de coût German Credit (Hofmann, 1994) : refuser un bon coûte 1, accepter un mauvais coûte 5.
COST_FP = 5.0   # coût d'approuver un mauvais emprunteur (faux positif)
COST_FN = 1.0   # coût de refuser un bon emprunteur (faux négatif)
TAU_OPT = COST_FP / (COST_FP + COST_FN)   # seuil cost-optimal = 5/6 ≈ 0.833
np.random.seed(SEED); sns.set_style("whitegrid"); plt.rcParams["figure.dpi"] = 100

# --- Données --------------------------------------------------------------
df = fetch_openml("credit-g", version=1, as_frame=True).frame.copy()
df["target"] = (df["class"] == "good").astype(int)
FEATURE_COLS = [c for c in df.columns if c not in ("class", "target")]
X_raw = df[FEATURE_COLS].copy()
y = df["target"].values
# Convention : 1 = adultes (favorisé), 0 = jeunes (défavorisé)
A = (df["age"].astype(float) >= AGE_THRESHOLD).astype(int).values

# Split stratifié sur (y, A)
strates = pd.Series(y).astype(str) + "_" + pd.Series(A).astype(str)
X_tr_raw, X_te_raw, y_tr, y_te, A_tr, A_te = train_test_split(
    X_raw, y, A, test_size=0.3, stratify=strates, random_state=SEED,
)

# --- Préprocesseur partagé -----------------------------------------------
cat_cols = X_tr_raw.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_tr_raw.select_dtypes(include=[np.number]).columns.tolist()
preproc = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), cat_cols),
    ("num", StandardScaler(), num_cols),
]).fit(X_tr_raw)

def preprocess_fn(df_human: pd.DataFrame) -> np.ndarray:
    return preproc.transform(df_human[FEATURE_COLS])

# --- Modèles -------------------------------------------------------------
clf_lr = Pipeline([
    ("prep", preproc),
    ("lr",  LogisticRegression(C=1e10, max_iter=5000, solver="lbfgs", random_state=SEED)),
]).fit(X_tr_raw, y_tr)

def predict_score(model, x_df: pd.DataFrame) -> np.ndarray:
    return model.predict_proba(x_df[FEATURE_COLS])[:, 1]

def get_group(x_df: pd.DataFrame) -> np.ndarray:
    return (x_df["age"].astype(float).values >= AGE_THRESHOLD).astype(int)

p_lr_te  = predict_score(clf_lr,  X_te_raw)

print(f"n_test = {len(y_te)} | LR AUC = {roc_auc_score(y_te, p_lr_te):.3f} | "
      f"Refusés à τ*={TAU_OPT:.3f} : {(p_lr_te < TAU_OPT).sum()}/{len(p_lr_te)}")

# === Module PACR-AP : on prépare juste le sys.path ; chaque cellule
# ci-dessous fait son propre reload + import (itération sans kernel restart).
from pathlib import Path
import importlib, pacr_ap
importlib.reload(pacr_ap)


In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)
from pacr_ap import GlobalThresholdRule, load_schema_from_yaml, compute_feature_scales, mine_actions_from_data, calibrate_epsilon, build_joint_plausibility_check, PlausibilityConfig, make_lr_pipeline_factory, BootstrapConfig, fit_bootstrap_models

# ─── Règle de décision : seuil cost-optimal ────────────────────────────────
rule_global = GlobalThresholdRule(tau=TAU_OPT, margin=0.0)

# ─── Schéma d'actionnabilité (variant conservateur) ─────────────────────────
POLICY_PATH = Path("schema-actions.yaml")
SCHEMA, ORDINAL_ORDERS, EMP_BOUNDS = load_schema_from_yaml(POLICY_PATH, variant_name="conservateur")

# ─── Calibration empirique (training set) ──────────────────────────────────
SCALES = compute_feature_scales(X_tr_raw, SCHEMA)

# ─── Actions élémentaires : magnitudes MINÉES depuis les données ────────────
# Pour chaque feature mutable, on apprend les magnitudes d'action depuis les
# écarts observés entre refusés et leurs voisins favorables dans le training.
ACTIONS, MINING_REPORT = mine_actions_from_data(
    X_tr_raw, y_tr, SCHEMA, ORDINAL_ORDERS, EMP_BOUNDS, SCALES,
    k_neighbors=10, min_support=30,
)

# ─── Plausibilité hybride : marginale (NN) + jointe (LOF) ──────────────────
EPSILON_NN = calibrate_epsilon(X_tr_raw, SCHEMA, SCALES, percentile=95.0)
is_jointly_plausible, joint_score_fn, RHO_LOF = build_joint_plausibility_check(
    X_tr_raw, preproc, FEATURE_COLS, n_neighbors=20, percentile=5.0,
)
PLAUS_CFG = PlausibilityConfig(
    epsilon_nn=EPSILON_NN, min_density=0,
    joint_check_fn=is_jointly_plausible, rho_lof=RHO_LOF,
)

# ─── Bootstrap LR pour la robustesse intra-modèle ──────────────────────────
make_lr_pipeline = make_lr_pipeline_factory(cat_cols, num_cols, C=1e10, max_iter=5000, seed=SEED)
BOOT_CFG = BootstrapConfig(n_bootstrap=10, model_factory=make_lr_pipeline,
                            decision_rule=rule_global, threshold_strategy="fixed")
BOOTSTRAP_LR = fit_bootstrap_models(X_tr_raw, y_tr, A_tr, BOOT_CFG)

# ─── Récapitulatif compact ────────────────────────────────────────────────
n_mut = sum(1 for s in SCHEMA.values() if s.mutable)
print(f"τ* = {rule_global.required_threshold():.3f}  |  "
      f"{n_mut}/{len(SCHEMA)} features mutables  |  {len(ACTIONS)} actions minées")
print(f"Plausibilité : ε = {EPSILON_NN:.3f}, ρ_LOF = {RHO_LOF:.3f}  |  "
      f"Bootstrap : {len(BOOTSTRAP_LR)} modèles LR")

# ─── Échantillon d'audit ÉQUILIBRÉ par groupe sensible ────────────────────
# Le déséquilibre naturel (~17 jeunes vs ~155 adultes refusés sous τ*) rendait
# les IC sur les gaps statistiquement indéterminés. On échantillonne 15 refusés
# par groupe (ou tous s'il y en a moins) pour rendre les IC comparables.
N_PER_GROUP = 15
_rng_audit = np.random.default_rng(SEED)
_refused_mask = p_lr_te < TAU_OPT
_refused_pos = np.where(_refused_mask)[0]
_ref_J = _refused_pos[A_te[_refused_pos] == 0]
_ref_A = _refused_pos[A_te[_refused_pos] == 1]
_samp_J = _rng_audit.choice(_ref_J, size=min(N_PER_GROUP, len(_ref_J)), replace=False)
_samp_A = _rng_audit.choice(_ref_A, size=min(N_PER_GROUP, len(_ref_A)), replace=False)
AUDIT_POSITIONS = np.sort(np.concatenate([_samp_J, _samp_A]))

X_te_audit = X_te_raw.iloc[AUDIT_POSITIONS].reset_index(drop=True)
A_te_audit = A_te[AUDIT_POSITIONS]
p_lr_audit = p_lr_te[AUDIT_POSITIONS]

print(f"Audit équilibré : {(A_te_audit == 0).sum()} jeunes + {(A_te_audit == 1).sum()} adultes")


## 4. Application sur un individu refusé

Illustration sur un jeune (A=0) refusé proche du seuil. PACR-AP construit son
graphe d'actions, sélectionne 3 chemins divers, et produit les explications
textuelles + le graphe + la progression du score.


In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)
from pacr_ap import GraphConfig, generate_action_path_recourse

# Choix d'un individu refusé sous TAU_OPT pour la démo.
# On prend le jeune (A=0) avec le score le plus élevé parmi les refusés —
# c'est le candidat le plus actionnable et le plus pédagogique.
refused_lr_idx = np.where(p_lr_te < TAU_OPT)[0]
young_refused = refused_lr_idx[A_te[refused_lr_idx] == 0]
if len(young_refused) > 0:
    DEMO_IDX = int(young_refused[np.argmax(p_lr_te[young_refused])])
else:
    DEMO_IDX = int(refused_lr_idx[np.argmax(p_lr_te[refused_lr_idx])])

x_demo = X_te_raw.iloc[DEMO_IDX]
# Le profil détaillé apparaît dans le titre du graphe (cellule plot_action_graph ci-dessous).

res_demo = generate_action_path_recourse(
    x_demo, clf_lr, rule_global, SCHEMA, ACTIONS, X_tr_raw,
    PLAUS_CFG, SCALES,
    graph_cfg=GraphConfig(max_depth=4, beam_width=60, max_nodes=1500),
    boot_models=BOOTSTRAP_LR, feature_cols=FEATURE_COLS, model_name="LR",
)
print(f"Graphe : {len(res_demo['graph']['nodes'])} nœuds  |  "
      f"{len(res_demo['all_valid_paths'])} favorables  |  "
      f"{len(res_demo['selected_paths'])} sélectionnés")


In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)
from pacr_ap import render_recourse_cards

# Cartes HTML : une par chemin, couleur synchro avec le graphe ci-dessous
render_recourse_cards(
    res_demo["selected_paths"], clf_lr, FEATURE_COLS, EPSILON_NN,
    n_bootstrap=len(BOOTSTRAP_LR),
    all_paths=res_demo["all_valid_paths"],
    model_name="LR",
)

In [ ]:
# Reload du module à chaque exécution (pour itérer sur le code de plot)
import importlib, pacr_ap
importlib.reload(pacr_ap)
from pacr_ap import plot_action_graph

# Visualisation du graphe d'actions local (fonction dans le module)
plot_action_graph(
    res_demo["graph"], res_demo["selected_paths"], rule_global,
    all_paths=res_demo["all_valid_paths"],   # → affiche Σ + rang dans la légende
    title=f"Graphe d'actions local — individu test #{DEMO_IDX} "
           f"(A={int(A_te[DEMO_IDX])}, score initial {p_lr_te[DEMO_IDX]:.3f}, LR)",
)


Le graphe explore toutes les façons dont cet individu peut faire basculer la décision, avec en couleur les trois pistes que la méthode retient. La courbe juste en dessous classe tous les candidats par score multi-objectif Σ pour montrer où se placent ces trois-là, et les trois scatters confirment le compromis axe par axe.

L'individu choisi pour la démo avait un score initial (0.826) qui frôlait déjà le seuil (0.833), ce qui explique pourquoi les chemins retenus sont très courts : une seule action — réduire la durée ou le montant du crédit — suffit à faire basculer la décision. La marge gagnée reste mince et la robustesse bootstrap modeste (~50 %), signe qu'on est vraiment à la limite.

## 5. Audit fairness par groupe

On applique le pipeline aux 30 refusés équilibrés (15 + 15), agrège les
métriques par groupe sensible, calcule les gaps $\Delta_M = M_{A=0} - M_{A=1}$
(jeunes − adultes), et estime leurs **intervalles de confiance bootstrap 95 %**
pour juger de la significativité statistique.


In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)
from pacr_ap import GraphConfig, generate_action_path_recourse_for_dataset

recourse_lr = generate_action_path_recourse_for_dataset(
    X_te_audit, A_te_audit, clf_lr, rule_global, SCHEMA, ACTIONS, X_tr_raw,
    PLAUS_CFG, SCALES,
    graph_cfg=GraphConfig(max_depth=4, beam_width=60, max_nodes=1500),
    boot_models=BOOTSTRAP_LR, feature_cols=FEATURE_COLS,
    model_name="LR", max_individuals=len(X_te_audit),
)
print(f"Coverage LR : {recourse_lr['has_path'].mean():.1%}  "
      f"(jeunes {recourse_lr[recourse_lr.group==0]['has_path'].mean():.0%}  /  "
      f"adultes {recourse_lr[recourse_lr.group==1]['has_path'].mean():.0%})")


In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)
from pacr_ap import fairness_of_recourse_paths

# `fairness_of_recourse_paths` est dans le module.
# Données pour les plots qui suivent — l'audit visuel (cell suivante)
# et les CI bootstrap consomment fair_lr et gaps_lr directement.
fair_lr, gaps_lr = fairness_of_recourse_paths(recourse_lr)



In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)
from pacr_ap import plot_audit_panels

# Audit visuel : coverage / coût / robustesse / plausibilité (médiane noire + points individuels)
plot_audit_panels(recourse_lr, fair_lr, EPSILON_NN,
                   title_prefix=f"LR · τ* = {TAU_OPT:.3f}")


Quatre vues côte à côte qui posent la question sous tous les angles : qui obtient un recourse, à quel coût, avec quelle robustesse au bruit du modèle, et avec quelle plausibilité de l'état final.

La coverage est strictement égale entre les deux groupes (67 % chacun), donc personne n'est exclu du recourse par construction. La disparité se joue ailleurs : le coût médian des jeunes est **5.24** contre **2.81** chez les adultes (presque le double), et leurs chemins survivent moins bien au bruit du modèle (médiane de robustesse **50 % vs 70 %**).

### 5.bis Quelles features sont modifiées par chaque groupe ?

Le tableau et les panneaux ci-dessus comparent les **métriques agrégées** (coverage, coût, robustesse, plausibilité). On regarde maintenant **quelles features** chaque groupe modifie effectivement dans son chemin top-1. Si jeunes et adultes ont un coût moyen identique mais touchent des features très différentes, le verdict gagne en nuance qualitative.

In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)
from pacr_ap import plot_feature_modification_dumbbell

# Heatmap : % d'individus du groupe modifiant chaque feature
plot_feature_modification_dumbbell(recourse_lr,
                                    title_prefix=f"LR · τ* = {TAU_OPT:.3f}")


Pour chaque feature actionnable, on lit côte à côte le pourcentage de jeunes (point orange) et d'adultes (point vert) qui la modifient dans leur chemin top-1. La longueur du segment entre les deux points donne directement l'écart entre groupes.

In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)
from pacr_ap import plot_gap_confidence_intervals

# Intervalles de confiance bootstrap sur les 3 gaps de fairness
# Réponse rigoureuse à la critique n=3 : les gaps sont-ils significatifs ?
_ = plot_gap_confidence_intervals(recourse_lr, n_bootstrap=500, seed=SEED,
                                   title_prefix=f"LR · τ* = {TAU_OPT:.3f}")


Avec seulement 15 jeunes et 15 adultes dans l'audit, les gaps moyens sont fragiles — il suffirait de retirer deux individus pour les voir bouger sensiblement. C'est précisément pour ça qu'on rejoue l'audit 500 fois par bootstrap : ré-échantillonner les individus avec remise permet d'estimer dans quelle fourchette chaque gap pourrait raisonnablement se trouver, et donc de distinguer un vrai écart entre groupes d'une simple fluctuation due à la petite taille de l'échantillon. Si l'intervalle traverse la ligne rouge (zéro), on ne peut pas conclure à une disparité ; sinon, le gap tient malgré le faible n.

Concrètement, les trois intervalles traversent zéro, donc strictement parlant aucun gap n'est significatif au seuil 5 %. Mais Δ_cost et Δ_robust ne le franchissent que d'un cheveu (bornes à −0.06 et +0.01) — avec un échantillon un peu plus grand, ils sortiraient très probablement comme significatifs.

## 6. Exploration interactive

Dashboard cliquable : un point = un refusé. Cliquer révèle son chemin top-1
— score à chaque étape + action appliquée + coût cumulé.


In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)
from pacr_ap import GraphConfig, generate_action_path_recourse, action_short_label

import altair as alt
alt.data_transformers.disable_max_rows()

# --- Reconstruction des chemins pas-à-pas pour chaque refusé ----------------
# (Le batch global ne stockait que les métriques top-1 ; ici on récupère
# les trajectoires complètes pour l'animation interactive.)
scatter_rows = []
step_rows = []

for _, r in recourse_lr.iterrows():
    test_idx = int(r["test_idx"])
    group_lbl = "Jeune (A=0)" if r["group"] == 0 else "Adulte (A=1)"
    score_orig = float(r["score_orig"])

    if not r["has_path"]:
        scatter_rows.append({
            "test_idx": test_idx, "group": group_lbl,
            "score_orig": score_orig, "final_score": score_orig,
            "cumulative_cost": 0.0, "path_length": 0,
            "robust_validity": 0.0, "has_path": "Aucun chemin trouvé",
            "first_action": "—", "action_sequence": "—",
        })
        step_rows.append({
            "test_idx": test_idx, "group": group_lbl,
            "step": 0, "score": score_orig,
            "action_short": "(initial — aucun chemin)", "action_full": "(aucun chemin trouvé)",
            "cost_cum": 0.0,
        })
        continue

    # Re-exécution pour récupérer la trajectoire complète
    x0 = X_te_audit.iloc[test_idx]
    res = generate_action_path_recourse(
        x0, clf_lr, rule_global, SCHEMA, ACTIONS, X_tr_raw,
        PLAUS_CFG, SCALES,
        graph_cfg=GraphConfig(max_depth=4, beam_width=60, max_nodes=1500),
        boot_models=BOOTSTRAP_LR, feature_cols=FEATURE_COLS, model_name="LR",
    )
    if not res["selected_paths"]:
        continue
    top = res["selected_paths"][0]

    scatter_rows.append({
        "test_idx": test_idx, "group": group_lbl,
        "score_orig": score_orig,
        "final_score": float(top.final_score),
        "cumulative_cost": float(top.cumulative_cost),
        "path_length": int(top.length),
        "robust_validity": float(top.robust_validity) if top.robust_validity is not None else 0.0,
        "has_path": "Chemin trouvé",
        "first_action": top.first_action or "—",
        "action_sequence": " → ".join(a.name for a in top.actions),
    })

    # Étape 0 = état initial
    step_rows.append({
        "test_idx": test_idx, "group": group_lbl,
        "step": 0, "score": score_orig,
        "action_short": "(initial)", "action_full": "(état initial du dossier)",
        "cost_cum": 0.0,
    })
    cum = 0.0
    for t, act in enumerate(top.actions, start=1):
        cum += act.cost_fn(top.states[t-1])
        score_t = float(predict_score(clf_lr, pd.DataFrame([top.states[t]]))[0])
        v_o = top.states[t-1][act.feature]
        v_n = top.states[t][act.feature]
        if isinstance(v_o, float):
            v_o_str = f"{v_o:.0f}"
            v_n_str = f"{v_n:.0f}"
        else:
            v_o_str = str(v_o)
            v_n_str = str(v_n)
        step_rows.append({
            "test_idx": test_idx, "group": group_lbl,
            "step": t, "score": score_t,
            "action_short": action_short_label(act),
            "action_full": f"{action_short_label(act)} — {act.feature}: {v_o_str} → {v_n_str}",
            "cost_cum": cum,
        })

scatter_df = pd.DataFrame(scatter_rows)
step_df = pd.DataFrame(step_rows)
print(f"Scatter : {len(scatter_df)} individus | Step chart : {len(step_df)} étapes")

# --- Dashboard Altair interactif --------------------------------------------
COLOR_SCALE = alt.Scale(
    domain=["Jeune (A=0)", "Adulte (A=1)"],
    range=["#E67E22", "#16A085"],
)
SHAPE_SCALE = alt.Scale(
    domain=["Chemin trouvé", "Aucun chemin trouvé"],
    range=["circle", "cross"],
)

# Sélection : clic sur un point
click_sel = alt.selection_point(fields=["test_idx"], on="click", empty=False, name="picker")

# Panel 1 : scatter score_orig vs final_score
scatter = alt.Chart(scatter_df).mark_point(size=200, filled=True, stroke="white", strokeWidth=1.5).encode(
    x=alt.X("score_orig:Q", title="Score initial f(x₀)",
             scale=alt.Scale(domain=[0, 1])),
    y=alt.Y("final_score:Q", title="Score final après recourse",
             scale=alt.Scale(domain=[0, 1])),
    color=alt.Color(
        "group:N", scale=COLOR_SCALE,
        legend=alt.Legend(
            title="Groupe sensible", orient="right",
            titleFontSize=12, labelFontSize=11,
            symbolSize=220, symbolStrokeWidth=0,
            padding=10, offset=10,
        ),
    ),
    shape=alt.Shape(
        "has_path:N", scale=SHAPE_SCALE,
        legend=alt.Legend(
            title="Statut", orient="right",
            titleFontSize=12, labelFontSize=11,
            symbolSize=180, padding=10, offset=10,
        ),
    ),
    opacity=alt.condition(click_sel, alt.value(1.0), alt.value(0.5)),
    size=alt.condition(click_sel, alt.value(450), alt.value(180)),
    tooltip=[
        alt.Tooltip("test_idx:Q", title="Individu #"),
        alt.Tooltip("group:N", title="Groupe"),
        alt.Tooltip("score_orig:Q", format=".3f", title="Score initial"),
        alt.Tooltip("final_score:Q", format=".3f", title="Score final"),
        alt.Tooltip("path_length:Q", title="Longueur du chemin"),
        alt.Tooltip("cumulative_cost:Q", format=".3f", title="Coût cumulé"),
        alt.Tooltip("robust_validity:Q", format=".0%", title="Robustesse bootstrap"),
        alt.Tooltip("first_action:N", title="Première action"),
        alt.Tooltip("action_sequence:N", title="Séquence d'actions"),
    ],
).add_params(click_sel).properties(
    width=620, height=420,
    title=alt.TitleParams(
        text="Cliquez sur un individu pour voir sa trajectoire de recourse",
        subtitle=[
            "Axe x = score avant recourse | Axe y = score après recourse",
            "Couleur = groupe sensible | Forme = recourse trouvé ou non",
        ],
        fontSize=14, anchor="start",
    ),
)

# Lignes de référence : diagonale (no change) + threshold τ*
diag = alt.Chart(pd.DataFrame({"x": [0, 1], "y": [0, 1]})).mark_line(
    color="#888", strokeDash=[3, 3]
).encode(x="x:Q", y="y:Q")

tau_h = alt.Chart(pd.DataFrame({"y": [TAU_OPT]})).mark_rule(
    color="#E74C3C", strokeDash=[6, 4], size=2
).encode(y="y:Q")

tau_v = alt.Chart(pd.DataFrame({"x": [TAU_OPT]})).mark_rule(
    color="#E74C3C", strokeDash=[6, 4], size=2, opacity=0.4
).encode(x="x:Q")

tau_label = alt.Chart(pd.DataFrame({"x": [0.02], "y": [TAU_OPT + 0.025], "text": ["τ* = 0.833"]})).mark_text(
    color="#E74C3C", fontSize=11, fontWeight="bold", align="left",
).encode(x="x:Q", y="y:Q", text="text:N")

scatter_layer = (scatter + diag + tau_h + tau_v + tau_label).interactive()

# Panel 2 : progression du score le long du chemin sélectionné
# Line + points
step_base = alt.Chart(step_df).transform_filter(click_sel)

step_line = step_base.mark_line(
    color="#9B59B6", strokeWidth=4, point=alt.OverlayMarkDef(
        size=220, filled=True, stroke="white", strokeWidth=2,
    )
).encode(
    x=alt.X("step:O", title="Étape du chemin"),
    y=alt.Y("score:Q", title="Score du modèle", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color("group:N", scale=COLOR_SCALE, legend=None),
    tooltip=[
        alt.Tooltip("step:O", title="Étape"),
        alt.Tooltip("score:Q", format=".3f", title="Score"),
        alt.Tooltip("action_full:N", title="Action"),
        alt.Tooltip("cost_cum:Q", format=".3f", title="Coût cumulé"),
    ],
)

# Étiquettes textuelles d'action sur chaque point
step_labels = step_base.mark_text(
    align="left", dx=12, dy=-12, fontSize=10.5, color="#1F2A36",
).encode(
    x="step:O", y="score:Q", text="action_short:N",
)

# Seuil τ*
step_tau = alt.Chart(pd.DataFrame({"y": [TAU_OPT]})).mark_rule(
    color="#E74C3C", strokeDash=[6, 4], size=2,
).encode(y="y:Q")

step_tau_label = alt.Chart(pd.DataFrame({
    "x": [0], "y": [TAU_OPT + 0.03], "text": ["τ* = 0.833 (seuil cost-optimal)"]
})).mark_text(color="#E74C3C", fontSize=11, fontWeight="bold", align="left").encode(
    x="x:O", y="y:Q", text="text:N",
)

step_chart = (step_line + step_labels + step_tau + step_tau_label).properties(
    width=620, height=320,
    title=alt.TitleParams(
        text="Progression du score le long du chemin sélectionné",
        subtitle="Chaque point = un état | étiquette = action appliquée pour y arriver",
        fontSize=13, anchor="start",
    ),
)

# Panel 3 : coût cumulé au fil des étapes
cost_chart = step_base.mark_area(
    color="#5AAC68", opacity=0.6, line={"strokeWidth": 3},
).encode(
    x=alt.X("step:O", title="Étape"),
    y=alt.Y("cost_cum:Q", title="Coût cumulé (σ-units)"),
    tooltip=[
        alt.Tooltip("step:O", title="Étape"),
        alt.Tooltip("cost_cum:Q", format=".3f", title="Coût cumulé"),
        alt.Tooltip("action_full:N", title="Action"),
    ],
).properties(
    width=620, height=200,
    title=alt.TitleParams(text="Coût cumulé du chemin",
                          subtitle="Sommé en multiples d'écart-type empirique",
                          fontSize=12, anchor="start"),
)

dashboard = scatter_layer & step_chart & cost_chart
dashboard


Tableau de bord interactif : chaque point représente un refusé, placé selon son score initial (axe x) et son score après recourse (axe y). Un clic sur un point fait apparaître en bas la trajectoire complète du chemin recommandé pour cet individu.

## 7. Robustesse du verdict au schéma d'actions

Le **schéma d'actionnabilité** (quelles features sont mutables) est le seul
choix politique non-empirique de la méthode. On vérifie que le verdict de
fairness ne dépend pas arbitrairement de ce choix, en rejouant l'audit sous
trois variants (`conservateur`, `modéré`, `permissif`). Un verdict défendable
garde le **même signe** sur les trois.


In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)
from pacr_ap import GraphConfig, load_schema_from_yaml, mine_actions_from_data, generate_action_path_recourse_for_dataset, fairness_of_recourse_paths

# Sensibilité au schéma : on relance l'audit sous chacun des 3 variants du YAML.
SENSITIVITY_VARIANTS = ["conservateur", "modere", "permissif"]

sensitivity_rows = []
for variant in SENSITIVITY_VARIANTS:
    schema_v, orders_v, _ = load_schema_from_yaml(POLICY_PATH, variant_name=variant)
    # Mining d'actions pour ce variant (cohérence avec §6)
    actions_v, _report_v = mine_actions_from_data(
        X_tr_raw, y_tr, schema_v, orders_v, EMP_BOUNDS, SCALES,
        k_neighbors=10, min_support=30,
    )
    print(f"  variant {variant:>13s} : {len(actions_v)} actions minées, {sum(1 for s in schema_v.values() if s.mutable)} features mutables")
    recourse_v = generate_action_path_recourse_for_dataset(
        X_te_audit, A_te_audit, clf_lr, rule_global, schema_v, actions_v, X_tr_raw,
        PLAUS_CFG, SCALES,
        graph_cfg=GraphConfig(max_depth=4, beam_width=60, max_nodes=1500),
        boot_models=BOOTSTRAP_LR, feature_cols=FEATURE_COLS,
        model_name="LR", max_individuals=len(X_te_audit),
    )
    fair_v, gaps_v = fairness_of_recourse_paths(recourse_v)
    sensitivity_rows.append({
        "variant": variant, "n_actions": len(actions_v),
        "n_J": int(fair_v.iloc[0]["n_refused"]), "n_A": int(fair_v.iloc[1]["n_refused"]),
        "cov_J": fair_v.iloc[0]["coverage"], "cov_A": fair_v.iloc[1]["coverage"],
        "cost_J": fair_v.iloc[0]["mean_cost"], "cost_A": fair_v.iloc[1]["mean_cost"],
        "robust_J": fair_v.iloc[0]["mean_robust"], "robust_A": fair_v.iloc[1]["mean_robust"],
        "Δ_cov": float(gaps_v.iloc[0, 0]) if len(gaps_v) else np.nan,
        "Δ_cost": float(gaps_v.iloc[0, 2]) if len(gaps_v) else np.nan,
        "Δ_robust": float(gaps_v.iloc[0, 6]) if len(gaps_v) else np.nan,
    })

sens_df = pd.DataFrame(sensitivity_rows)
# Tableau complet disponible via sens_df — visualisé dans la cellule suivante.


In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)
from pacr_ap import plot_sensitivity_gaps

plot_sensitivity_gaps(
    sens_df,
    title="Sensibilité du verdict au schéma d'actions",
    context=f"LR · τ* = {TAU_OPT:.3f}",
)


On rejoue l'audit sous chacun des trois variants du schéma d'actions (`conservateur`, `modéré`, `permissif`) et on compare les trois gaps de fairness obtenus : une barre par couple métrique × variant.

**Lecture du verdict.** Pour chaque métrique (`Δ_coverage`, `Δ_cost`, `Δ_robust`), on regarde le **signe** des barres à travers les 3 variants :

- **signe identique sur les 3** → le verdict est **robuste** au choix de politique : le gap entre jeunes et adultes existe quel que soit le schéma d'actionnabilité retenu, ce n'est pas un artefact de la politique.
- **signe qui s'inverse** → le verdict **dépend du schéma** : la conclusion sur la fairness change selon ce qu'on autorise comme actions. Il faut alors justifier explicitement le schéma choisi avant de conclure.

Concrètement ici, sur les 3 variants `conservateur` / `modéré` / `permissif`, le signe de `Δ_cost` reste **positif** : les jeunes paient systématiquement plus cher leur recourse que les adultes, indépendamment du périmètre d'actions autorisé. Le verdict de disparité tient.

## 8. Conclusion et limites

### Ce qu'on observe

| Gap | Médiane bootstrap | IC 95 % | Signes sur les 3 variants |
|---|---|---|---|
| Δ_coverage | +0.01 | [−0.29, +0.38] | 0 / +0 / 0 |
| Δ_cost | **+2.07** | [−0.06, +3.90] | +2.14 / +1.36 / +1.34 (tous positifs) |
| Δ_robust | **−0.12** | [−0.25, +0.01] | −0.12 / −0.16 / −0.06 (tous négatifs) |

Si on s'en tient aux IC bootstrap seuls, aucun des trois gaps n'est strictement significatif au seuil 5 % — tous traversent zéro d'un cheveu, et avec seulement 15 individus par groupe, c'est attendu. Mais sortir du strict cadre statistique pour regarder la **cohérence des signes** à travers les trois variants de schéma raconte une autre histoire : les jeunes paient systématiquement plus cher leur recourse (Δ_cost > 0 sur les trois variants) et leurs chemins sont systématiquement moins robustes au bruit du modèle (Δ_robust < 0 sur les trois). La coverage, elle, ne montre pas d'effet net.

Le bilan honnête : on a une **tendance solide** sur deux axes — coût et robustesse — confirmée par le stress-test au §7, mais pas un **fait certain** au sens strict. Le dataset est juste trop petit pour trancher sans hésitation, et il faudrait passer ces conclusions à un échantillon plus large avant d'en faire une affirmation forte.

### Ce que cette méthode ne fait pas (et pourquoi)

D'abord, le recourse reste **prédictif et pas causal** : on dit comment changer la décision du modèle, pas comment réduire le vrai risque de défaut (cf. Karimi et al. 2021). Réduire son montant de crédit fait baisser le score, mais ne rend pas l'individu objectivement meilleur emprunteur.

Ensuite, le **dataset est petit** — 1000 individus dont environ 17 jeunes refusés sous un seuil cost-optimal élevé. Même avec un sampling équilibré 15 + 15, les IC bootstrap restent larges et tirer une conclusion ferme demanderait beaucoup plus de données.

Le **schéma d'actionnabilité reste subjectif** : c'est un choix éthique-métier sur ce qu'on autorise un emprunteur à modifier. Notre stress-test §7 montre que le verdict tient sur trois variants raisonnables, mais quelqu'un avec un schéma franchement différent pourrait conclure autrement.

Il y a aussi une **dépendance circulaire** entre le mining et le modèle : les magnitudes d'actions sont apprises sur des transitions refusé → favorable du training, donc co-adaptées au modèle qu'on audite. Pour auditer un autre modèle, il faudrait re-miner les magnitudes depuis zéro.

Enfin, la **robustesse** mesurée est intra-modèle : le bootstrap teste la sensibilité au re-sampling du training, pas au choix de la classe de modèle. Un LR et un XGBoost pourraient bien proposer des chemins très différents pour le même individu.

Dernier point — toutes les actions sont supposées **indépendantes du temps**. La méthode raisonne sur des transitions instantanées et ne capture pas la dynamique réelle d'un parcours : *« d'abord A, puis B six mois plus tard »*, ni les éventuels effets d'apprentissage en cours de chemin. C'est une vraie limite quand on veut traduire un recourse en feuille de route concrète pour le client.

### Où aller ensuite

Trois pistes naturelles pour prolonger le travail :

- **Plausibilité plus riche** via des méthodes à variété latente comme C-CHVAE ou REVISE, plutôt que la combinaison NN + LOF qui reste superficielle pour des distributions complexes.
- **Recourse vraiment causal** à la Karimi et al. 2021, en posant un SCM explicite — bien plus exigeant en hypothèses, mais qui résoudrait la limite n°1.
- **Recourse multi-modèle** : ne retenir que les chemins qui font basculer plusieurs modèles à la fois (LR, XGBoost, RandomForest) — un recourse "consensus" plus défendable que celui qui ne tient que sur un seul classifieur.


---

# Annexes

Pour le lecteur curieux : les développements détaillés qui seraient noyés
dans le cœur du notebook.

## Annexe A. Spécification formelle détaillée


## Annexe A. Construction de PACR-AP, brique par brique

*La construction pas-à-pas en 9 étapes, avec définitions formelles et interprétations.*

Construisons la méthode brique par brique:

### A.1 Étape 1 — Définir ce que signifie « refusé »

**Problème** : on a besoin d'un critère sans ambiguïté pour dire qu'un individu
est refusé (et donc candidat au recourse) ou favorable.

**Réponse** : une **règle de décision**

$$D : [0, 1] \times \{0, 1\} \to \{0, 1\}$$

qui prend le score $s$ et le groupe sensible $a$, et retourne 0 (refusé) ou
1 (favorable). PACR-AP est **agnostique** au choix de $D$ — seuil global,
seuil par groupe, post-processing Hardt, règle custom : tout fonctionne. Le
choix spécifique pour nos expériences est discuté au §6.

### A.2 Étape 2 — Restreindre ce qu'on peut modifier

**Problème** : on ne peut pas demander à un individu de *« changer d'âge »*.
Il faut déclarer explicitement quelles features sont modifiables, dans quel
sens, et entre quelles valeurs.

**Réponse** : un **schéma d'actionnabilité** $S$ qui annote chaque feature $f$

$$S(f) = (\text{mutable}_f,\; \text{direction}_f,\; \text{type}_f,\; \text{valeurs\_autorisées}_f)$$

externalisé dans un fichier YAML pour rester négociable par des stakeholders
non-développeurs. Trois variants (`conservateur`, `modéré`, `permissif`) seront
utilisés pour stress-tester le verdict (§10).

### A.3 Étape 3 — Donner un sens à « modifier »

**Problème** : modifier comment ? un déplacement continu dans l'espace ?
Difficile à interpréter et à expliquer à un utilisateur. Mieux vaut un petit
ensemble d'opérations métier discrètes.

**Réponse** : appelons **profil** un vecteur de features d'un individu (montant,
durée, statut d'épargne, etc.) ; l'espace de tous les profils possibles est
noté $\mathcal{X}$. Un profil particulier est noté $\mathbf{x} \in \mathcal{X}$.

Une **action élémentaire** est alors un triplet

$$u = (T_u,\; c_u,\; \pi_u)$$

qui comprend :

- $T_u : \mathcal{X} \to \mathcal{X}$ — la **transformation** (par ex. *« réduire le montant de 10 % »*),
- $c_u(\mathbf{x}) \in \mathbb{R}_+$ — le **coût empirique** depuis l'état $\mathbf{x}$,
- $\pi_u(\mathbf{x}) \in \{0, 1\}$ — la **précondition** (l'action est-elle applicable depuis $\mathbf{x}$ ?).

Le coût $c_u(\mathbf{x})$ est **calibré sur le training**, selon le type de la
feature modifiée :

- **Feature continue** (par ex. montant, durée) :
  $$c_u(\mathbf{x}) = \frac{|\Delta|}{\sigma_f}$$
  où $\Delta$ est l'écart appliqué par l'action (par ex. $0{,}10 \times x_{\text{montant}}$
  pour une réduction de 10 %), et $\sigma_f$ est l'**écart-type empirique** de
  la feature $f$ dans le training. *Interprétation* : « cette action vaut $k$
  écarts-types de la population observée ».

- **Feature ordinale ou catégorielle** :
  $$c_u(\mathbf{x}) = -\log \widehat{P}(v_{\text{cible}})$$
  où la **fréquence empirique** de la valeur cible dans le training est
  $$\widehat{P}(v_{\text{cible}}) = \frac{\#\{i \in \text{train} : x_i^f = v_{\text{cible}}\}}{n_{\text{train}}}$$
  c'est-à-dire la proportion d'individus du training ayant exactement cette
  valeur pour la feature $f$. (Pour les ordinales, on utilise
  $\widehat{P}(x_i^f \geq v_{\text{cible}})$ : proportion d'individus *au moins
  au niveau cible*.)

  *Pourquoi $-\log \widehat{P}$ ?* C'est la **self-information** de Shannon : un
  évènement rare porte plus d'information qu'un évènement fréquent. L'unité
  s'appelle le **nat** (logarithme naturel) — équivalent au *bit* mais en base
  $e$ au lieu de base $2$. Concrètement, si 10 % des individus ont la valeur
  cible, $c_u = -\log 0{,}1 \approx 2{,}3$ nats ; si 50 % l'ont,
  $c_u \approx 0{,}7$ nats. Plus la cible est rare, plus l'action coûte cher.

### A.4 Étape 4 — Explorer les chemins atteignables

**Problème** : combiner plusieurs actions pour basculer la décision demande
d'explorer un espace de séquences. Naïvement, l'espace explose.

**Réponse** : pour chaque refusé $\mathbf{x}_0$, on construit un **graphe
d'actions local**

$$G_{\mathbf{x}_0} = (V, E)$$

par BFS + beam search : à chaque profondeur, on ne garde que les
$K_{\text{beam}}$ états les plus prometteurs (par marge au seuil), jusqu'à
profondeur $T_{\max}$.

> **Notez** : les nœuds **ne sont pas des individus observés** (différence avec
> FACE), mais des profils synthétiques engendrés par application d'actions.
> Cette flexibilité a un prix — il faut filtrer les états implausibles à l'étape suivante.

### A.5 Étape 5 — Filtrer les états implausibles

**Problème** : un chemin techniquement valide peut passer par des états
absurdes (par ex. un profil financier incohérent). On veut exiger que
**toute la trajectoire** soit empiriquement crédible, pas seulement l'endpoint.

**Réponse** : chaque état $\mathbf{x}_t$ doit passer **deux filtres** conjoints :

$$
\text{plausible}(\mathbf{x}_t) \iff
\underbrace{d_{\text{NN}}(\mathbf{x}_t) \leq \varepsilon}_{\text{marginal}}
\;\;\text{ET}\;\;
\underbrace{\text{LOF}(\mathbf{x}_t) \geq \rho_{\text{LOF}}}_{\text{joint}}
$$

**Filtre marginal — $d_{\text{NN}}(\mathbf{x}_t)$** : distance au point le plus
proche du training,

$$d_{\text{NN}}(\mathbf{x}_t) = \min_{\mathbf{z} \in \mathcal{D}_{\text{train}}} d(\mathbf{x}_t, \mathbf{z})$$

avec $d$ = distance L1 mixte : pour une feature continue $f$ on ajoute
$|x_t^f - z^f| / \sigma_f$, pour une catégorielle on ajoute
$\mathbb{1}\{x_t^f \neq z^f\}$. Si cette distance est petite, $\mathbf{x}_t$
ressemble à un individu réellement observé.

**Filtre joint — $\text{LOF}(\mathbf{x}_t)$** : on compare la densité locale
autour de $\mathbf{x}_t$ à celle autour de ses voisins,

$$\text{LOF}_k(\mathbf{x}_t) \;\sim\; \frac{\overline{\text{densité}}\bigl(\mathbf{y} \in \mathcal{N}_k(\mathbf{x}_t)\bigr)}{\text{densité}(\mathbf{x}_t)}$$

où $\mathcal{N}_k(\mathbf{x}_t)$ est l'ensemble des $k$ voisins les plus proches
dans le training, et la *densité locale* d'un point est l'inverse de sa distance
moyenne à ses propres $k$ voisins. *Intuition* : si $\mathbf{x}_t$ est dans une
zone aussi dense que celle de ses voisins, c'est une région typique ; s'il est
isolé alors que ses voisins sont denses ailleurs, c'est un **outlier conjoint**
(combinaison rare).

> **Convention sklearn** : on utilise l'opposé du LOF canonique de Breunig
> (`negative_outlier_factor`), donc plus le score est **haut**, plus le point
> est typique. Le seuil $\rho_{\text{LOF}}$ est calibré comme le **percentile 5**
> des scores intra-train ; $\varepsilon$ comme le **percentile 95** des
> distances NN intra-train. Un état passe les deux filtres s'il est au moins
> aussi typique que 95 % du training sur les deux dimensions.

### A.6 Étape 6 — Définir un chemin faisable

**Problème** : avec les notions précédentes, on peut maintenant assembler la
définition complète d'un chemin de recourse.

**Réponse** : un **chemin** est $P = (\mathbf{x}_0, \mathbf{x}_1, \ldots, \mathbf{x}_T)$
avec $\mathbf{x}_t = T_{u_t}(\mathbf{x}_{t-1})$. Il est **faisable** ssi :

1. **Validité finale** : $D(f(\mathbf{x}_T), a) = 1$ (Étape 1) ;
2. **Actionnabilité** : chaque $u_t$ respecte $S$ (Étape 2) ;
3. **Plausibilité de trajectoire** : chaque $\mathbf{x}_t$ ($t \geq 1$) passe le
   double filtre (Étape 5) ;
4. **Profondeur bornée** : $T \leq T_{\max}$.

### A.7 Étape 7 — Préférer les chemins robustes au modèle

**Problème** : un chemin valide sur notre modèle peut basculer en défavorable
sur un modèle équivalent (sensibilité au train sampling). On veut une mesure
de cette stabilité.

**Réponse** : on entraîne $B$ modèles bootstrap $f_1, \ldots, f_B$ — **du
même type que le modèle principal** (même architecture, mêmes hyperparamètres),
mais chacun ajusté sur un **rééchantillonnage avec remise** du training set
(taille $n$ identique, mais certains individus apparaissent en double, d'autres
pas du tout). Les coefficients diffèrent légèrement → la frontière de décision
est légèrement déplacée d'un modèle à l'autre.

Pour un état final $\mathbf{x}_T$ :

$$\text{RobustValidity}(\mathbf{x}_T) = \frac{1}{B} \sum_{b=1}^{B} \mathbb{1}\{D(f_b(\mathbf{x}_T), a) = 1\}$$

= fraction des modèles bootstrap qui maintiennent la décision favorable.
Un chemin est **robuste** ssi cette fraction dépasse un seuil $\rho_b$
(e.g. 0.8 = au moins 8 modèles sur 10 d'accord).

> **Ce que ça vérifie** : la sensibilité au **train sampling** uniquement
> — *« si on avait vu un échantillon légèrement différent du même processus, le
> chemin tiendrait-il toujours ? »*. Ça ne dit rien sur la robustesse à un
> changement de **classe de modèle** (LR → XGBoost → RF). Limite explicite,
> reconnue au §11.

### A.8 Étape 8 — Sélectionner K chemins divers à montrer

**Problème** : plusieurs chemins faisables et robustes existent. Lesquels
montrer à l'utilisateur ? Un seul est insuffisant ; trois variantes triviales
du même chemin sont redondantes.

**Réponse** : on note chaque chemin $P$ par un **objectif scalaire** (lower is
better). Pour cela, définissons d'abord les six quantités mesurées sur $P$ :

| Quantité | Formule |
|---|---|
| **Coût cumulé** $c(P)$ | $\sum_{t=1}^{T} c_{u_t}(\mathbf{x}_{t-1})$ — somme des coûts des actions (Étape 3) |
| **Longueur** $T(P)$ | nombre d'actions du chemin |
| **Sparsité** $s(P)$ | $\lvert \{f : \exists t,\; u_t \text{ modifie } f\} \rvert$ — nombre de features distinctes touchées |
| **Plausibilité worst** $d^{\max}_{\text{NN}}(P)$ | $\max_{t=0,\ldots,T} d_{\text{NN}}(\mathbf{x}_t)$ — distance NN du pire état de la trajectoire |
| **Marge finale** $m(P)$ | $f(\mathbf{x}_T) - \tau$ — de combien le score final dépasse le seuil |
| **Robustesse** $\rho(P)$ | $\text{RobustValidity}(\mathbf{x}_T)$ de l'Étape 7 |

Chacune est ensuite **normalisée min-max** sur l'ensemble des chemins candidats
pour ce refusé : $\tilde{q} = (q - q_{\min}) / (q_{\max} - q_{\min}) \in [0, 1]$.
On combine ensuite linéairement :

$$\text{Obj}(P) = \lambda_c \tilde{c} + \lambda_T \tilde{T} + \lambda_s \tilde{s} + \lambda_p \tilde{d}^{\max}_{\text{NN}} - \lambda_m \tilde{m} - \lambda_r \tilde{\rho}$$

Les quantités à **minimiser** (coût, longueur, sparsité, plausibilité worst)
ont un signe $+$ ; celles à **maximiser** (marge, robustesse) ont un signe $-$.

**Comment sont fixés les $\lambda$ ?** Ce ne sont **pas des grandeurs calculées**
mais des **hyperparamètres choisis manuellement** pour refléter les priorités
d'audit. Par défaut :

| Coefficient | Valeur | Interprétation |
|---|---|---|
| $\lambda_c,\; \lambda_p$ | $1{,}0$ | coût et plausibilité de trajectoire = priorités principales |
| $\lambda_T,\; \lambda_m,\; \lambda_r$ | $0{,}5$ | longueur, marge, robustesse = priorités secondaires |
| $\lambda_s$ | $0{,}3$ | sparsité tertiaire |

Modifiables via `PathScoringConfig` du module.

> $\lambda_p$ porte sur la plausibilité **de la trajectoire entière** 
> (worst state) via $d^{\max}_{\text{NN}}$, pas seulement de l'endpoint 
> — c'est ce qui distingue PACR-AP des méthodes classiques.

Enfin, on sélectionne les $K$ chemins de plus faible $\text{Obj}(P)$ **avec
contrainte de diversité**. Pour deux chemins $P_1, P_2$ d'ensembles de features
modifiées $\mathcal{F}_1, \mathcal{F}_2$, la **distance Jaccard** se définit par

$$d_J(P_1, P_2) = 1 - \frac{\lvert \mathcal{F}_1 \cap \mathcal{F}_2 \rvert}{\lvert \mathcal{F}_1 \cup \mathcal{F}_2 \rvert}$$

- $d_J = 0$ si les deux chemins touchent exactement les mêmes features,
- $d_J = 1$ si aucune feature commune.

On exige $d_J \geq 0{,}5$ entre tout
nouveau chemin sélectionné et tous les déjà retenus — pour éviter $K$ variantes
triviales du même chemin.

### A.9 Étape 9 — Auditer la fairness du recourse par groupe

**Problème** : la méthode produit des chemins individu par individu. Pour
juger de l'équité, il faut agréger par groupe sensible et comparer.

**Réponse** : pour chaque groupe $a \in \{0, 1\}$ on calcule
$\text{coverage}(a)$, $\overline{c}(a)$, $\overline{T}(a)$, $\overline{\rho}(a)$,
$\overline{d^{\max}_{\text{NN}}}(a)$, puis les **gaps** dans le sens conventionnel :

$$\Delta_M = M_{A=0} - M_{A=1} \quad \text{(jeunes − adultes)}$$

Un $\Delta_{\text{cost}} > 0$ veut dire *« les jeunes paient plus »* ; un
$\Delta_{\text{robust}} < 0$ veut dire *« leurs chemins sont moins stables »*.
Cette dimension est **complémentaire** à DP/EOpp/PP (équité des décisions vs
équité du recourse).

---

Voilà : avec ces neuf étapes, la méthode est entièrement définie.


## Annexe B. Littérature et emprunts détaillés


*Les 5 notions du recourse contrefactuel, chacune avec sa source littéraire,*
*plus les méthodes dont on s'est inspiré sans les emprunter.*

Le **recourse contrefactuel** répond à la question pratique :

> *« Mon dossier de crédit a été refusé par le modèle. Que dois-je changer pour qu'il soit accepté ? »*

Cinq notions sont nécessaires pour y répondre rigoureusement. Pour chacune,
on donne la définition puis la référence qui l'a introduite — c'est l'idée qu'on
emprunte à cette source.

### Notion 1 — Contrefactuel basculant
Un profil hypothétique modifié à partir de l'individu refusé, qui obtiendrait
une décision favorable du modèle. *Exemple : « si vous demandiez 5 000 DM au lieu
de 7 000, votre score passerait au-dessus du seuil ».*<br>
→ **Wachter, Mittelstadt & Russell (2017)** — *Counterfactual Explanations*.
On emprunte la **définition de la cible**.

### Notion 2 — Actionnabilité
Toutes les modifications ne sont pas autorisées : l'âge, le sexe, l'historique
ne peuvent ni ne doivent être changés. Un **schéma déclaratif** sépare ce qui
est **mutable** (montant, durée, épargne…) de ce qui est **immutable** (sensibles,
historiques).<br>
→ **Ustun, Spangher & Liu (2019)** — *Actionable Recourse*. On emprunte le
**principe du schéma par feature**.

### Notion 3 — Diversité multi-contrefactuels
Un seul contrefactuel optimal est insuffisant : l'utilisateur bénéficie de
**plusieurs alternatives** pour choisir celle qui matche son contexte
(capacités, contraintes personnelles).<br>
→ **Mothilal, Sharma & Tan (2020)** — *DiCE*. On emprunte la **sélection top-K
diverse**.

### Notion 4 — Chemin d'actions vs saut unique
Plutôt qu'un saut atomique *« passez de $x_0$ à $x_T$ »*, on décompose le recourse
en une **séquence d'actions élémentaires** : *« d'abord réduisez le montant de
10 %, puis améliorez votre épargne d'un cran »*. Chaque action est explicite,
chaque état intermédiaire doit rester réaliste.<br>
→ **Karimi, Barthe, Balle & Valera (2020)** — *MACE* pour l'**énumération discrète
d'actions** ; **Poyiadzi et al. (2020)** — *FACE* pour la **plausibilité de
trajectoire** (pas seulement de l'endpoint).

### Notion 5 — Plausibilité hybride
Un contrefactuel valide mathématiquement peut être empiriquement absurde (par ex.
montant élevé + engagement très faible). On exige donc que chaque état (a) **ressemble**
à des individus observés du training (proximité marginale) **ET** (b) ne forme pas
une **combinaison rare** de features (densité jointe).<br>
→ Distance marginale héritée du recourse classique. Combinaisons rares détectées
via **LOF — Breunig et al. (2000)**. On emprunte les **deux filtres conjoints**.

### Ce qu'on n'emprunte pas

- **FACE — chemins entre individus observés uniquement** : trop restrictif. On accepte des profils synthétiques tant qu'ils passent nos filtres.
- **C-CHVAE et REVISE — modèles génératifs appris (autoencodeurs variationnels)** : coûteux à entraîner et moins interprétables. LOF capture une part suffisante du bénéfice à coût marginal.
- **DiCE — diversité par DPP (Determinantal Point Process)** : élégant mais complexe à implémenter et à interpréter. On utilise une sélection gloutonne par distance Jaccard sur les features modifiées — plus simple, suffisant pour $K = 3$.
- **Karimi, Schölkopf & Valera (2021) — Causal Recourse (SCM complet)** : demande un modèle causal structurel non disponible pour German Credit (cf. §1 hors scope). On retient néanmoins leur **principe de garde-fou** : nos explications textuelles disent toujours *« cela change la décision du modèle »*, jamais *« cela vous rendra meilleur risque »* — pas une méthode empruntée, juste une discipline méthodologique.


## Annexe C. Mining des magnitudes depuis le training set

Cette annexe détaille comment les magnitudes d'actions sont **apprises depuis
les données** plutôt que choisies à la main. C'est ce qui permet à la méthode
d'éviter des actions arbitraires (par ex. *« réduire de 10 % »* sorti du
chapeau) et de coller à ce qu'on observe vraiment chez les emprunteurs qui
sont passés de refusé à favorable.

### Idée

L'intuition est simple : si on veut savoir de combien il *faut typiquement*
réduire un montant pour basculer une décision, le mieux est de regarder
**de combien les refusés diffèrent de leurs voisins favorables les plus proches**
dans le training. Ces écarts observés deviennent nos magnitudes candidates.

### Pipeline pas-à-pas

1. **Split du training** en deux sous-ensembles :
   - $\mathcal{R} = \{x_i : y_i = 0\}$ — les refusés
   - $\mathcal{F} = \{x_j : y_j = 1\}$ — les favorables

2. **Distance mixte** entre chaque refusé et chaque favorable. Pour une feature
   continue $f$, on standardise par l'écart-type : $|x_i^f - x_j^f| / \sigma_f$ ;
   pour une catégorielle, on ajoute $\mathbb{1}\{x_i^f \neq x_j^f\}$. Tout
   est sommé en distance L1.

3. **k plus proches voisins favorables** pour chaque refusé (k = 10 ici).
   On obtient donc $|\mathcal{R}| \times k$ paires *(refusé, voisin favorable)*.

4. **Collecte des écarts feature par feature**, en respectant la **direction
   du schéma** (decrease_only / increase_only) :
   - **continue** : on enregistre le pourcentage de variation
     $\text{pct} = |v_f - v_r| / |v_r|$
   - **ordinale** : on enregistre le nombre de crans $\text{step} = \text{idx}(v_f) - \text{idx}(v_r)$
   - **catégorielle** : on enregistre la valeur cible $v_f$

5. **Filtre de support** : une feature n'est éligible au mining que si elle a
   accumulé au moins `min_support = 30` observations à travers toutes les paires.
   Ça écarte les coïncidences statistiques.

6. **Discrétisation en actions canoniques** :
   - **continue** : on prend les percentiles 25 / 50 / 75 des `pct` observés,
     bornés à $[2\,\%, 50\,\%]$ pour éviter les actions triviales ou irréalistes.
   - **ordinale** : on retient les 2 valeurs de step les plus fréquentes,
     avec $|\text{step}| \leq 2$ et un comptage suffisant.
   - **catégorielle** : on retient les 3 valeurs cibles les plus fréquentes.

### Ce qu'on obtient sur German Credit

La cellule suivante imprime `MINING_REPORT` qui contient, pour chaque feature
mutable du schéma `conservateur` : le nombre de transitions observées, le
résumé des percentiles, et les magnitudes finalement retenues comme actions.

> **À garder en tête** : les transitions observées sont des **corrélations**,
> pas des **interventions** au sens causal. Le mining ancre les magnitudes
> dans le réel mais ne corrige pas la non-causalité fondamentale du recourse
> prédictif (cf. §8 et Hors scope).


In [ ]:
# Reload + import (itération sans kernel restart)
import importlib, pacr_ap; importlib.reload(pacr_ap)

# MINING_REPORT a été calculé dans la cellule 5 (setup) et reste dans le
# namespace. On l'affiche ici en clair pour rendre le mining auditable.
print(f"Refusés vus       : {MINING_REPORT['n_refused']}")
print(f"Favorables vus    : {MINING_REPORT['n_favorable']}")
print(f"k voisins / refusé: {MINING_REPORT['k_neighbors']}")
print(f"Min support       : {MINING_REPORT['min_support']}")
print(f"Total paires      : {MINING_REPORT['n_pairs_total']}")
print()
for feat, rep in MINING_REPORT['per_feature'].items():
    if 'skip' in rep:
        print(f"  {feat:<22s} : SKIPPED ({rep['skip']})")
        continue
    if not rep['actions_mined']:
        print(f"  {feat:<22s} : aucune action minée (filtres trop stricts)")
        continue
    print(f"  {feat:<22s} ({rep['type']}, {rep['direction']})")
    print(f"    └─ {rep['n_observations']} transitions observées")
    if rep['type'] == 'continuous':
        s = rep['observed_pct_summary']
        print(f"    └─ percentiles observés : p25={s['p25']*100:.0f}%  "
              f"p50={s['p50']*100:.0f}%  p75={s['p75']*100:.0f}%")
        vals = ", ".join(f"{a['value']*100:.0f}%" for a in rep['actions_mined'])
        print(f"    └─ magnitudes retenues  : [{vals}]")
    elif rep['type'] == 'ordinal':
        steps = ", ".join(f"step={a['value']:+d} (n={a['count']})"
                          for a in rep['actions_mined'])
        print(f"    └─ steps retenus        : {steps}")
    else:
        targets = ", ".join(f"{a['value']!r} (n={a['count']})"
                            for a in rep['actions_mined'])
        print(f"    └─ cibles retenues      : {targets}")
    print()
